# Flowsheet Inspector Library API

This section describes the application programming interface (API) used to
"wrap" flowsheets to make them structured and also allow them to be 
executed and return information through a well-known interface. This 
interface is used by the Flowsheet Inspector (FI). For information about how
to run these wrapped flowsheets see the [Usage](usage.md) section of the documentation.

## Contents

- [Wrapper API](#Simple-wrapper-API) - Simple single-function wrapper
- [Class API](#Class-based-API) - More flexible structured flowsheet wrapper
- [Actions](#Actions) - Framework for performing actions before and after flowsheet steps
- [Annotation](#Annotation) - Simple annotation capability (experimental)

## Terminology
There are some terms used below that have specific meanings.

- **action**: A way of gathering information (with possible side-effects) during a run. Actions are invoked before and after each step, and before and after the run, to execute arbitrary code; also produces a report with information gathered during the run.
- **flowsheet**: An IDAES flowsheet block
- **step**: One distinct part of building and solving a flowsheet, such as "build", "initialize", or "solve_optimization"
- **structured flowsheet**: The word "structured" refers to the use of a pre-defined order of *steps*, and the use of pre-defined *actions* invoked before/after steps and entire runs.
- **wrap**: To "wrap" a flowsheet means to add function decorators from the Flowsheet Inspector API to its steps (or its main function)


----

## Simple wrapper API

The structured flowsheet "simple wrapper" API allows existing flowsheet
scripts to be integrated into the Flowsheet Inspector with the addition
of an import and a single decorated `main()` function, which may already exist.


### Usage

The functionality of the API is imported with the name `fi_main`
in the `idaes_fi.structfs` package, so normal usage requires only a
single function, listed as `my_main_function()` in the example below
(some extra classes and functions are added so this can be a self-contained and
working example):

In [36]:

from idaes_fi.structfs import fi_main

@fi_main()
def my_main_function(some, args, keyword=None): # can take any arguments
    # build the flowsheet -> model
    model = build_flowsheet()
    # initialize the flowsheet
    # solve the flowsheet -> solve_result
    solve_result = solve_flowsheet()

    # **Important!**: return the model and solve result as a tuple
    return model, solve_result




Some classes so the build/solve can nominally succeed:

In [37]:
class FakeFlowsheet:
    is_indexed = lambda x: False
    def component_data_objects(self, *arg, **kw):
        return []
    def component_objects(self, *arg, **kw):
        return []

class FakeModel:
    fs = FakeFlowsheet()
    def component_objects(self, *arg, **kw):
        return []

# Fake build and solve functions

def build_flowsheet():
    # Fake build of flowsheet
    return FakeModel()

def solve_flowsheet():
    # Fake solve of flowsheet
    return {}


So, in summary, the steps to enable your flowsheets are:

1. Create a function that returns the tuple `(model, solve_result)` after
    building and solving the model.

2. Add the import statement `from idaes_fi.structfs import fi_main` and
    then decorate the function in (1) with  `@fi_main`

That's it! Now the Flowsheet Inspector can run your flowsheet and show the diagram,
model variables, degrees of freedom, diagnostics, etc.


In [38]:
fi_main()

<function idaes_fi.structfs.simple_wrap._Wrapper.main.<locals>.fi_wrapper_factory(main_fn)>


The rest of this section provides details on the more complex, but more
flexible, class-based API.

----

## Class-based API

The *struct*ured *f*low*s*heet runner is an API in the
{py:mod}`structfs <idaes_fi.structfs>` subpackage, and in
particular that package's {py:mod}`runner <idaes_fi.structfs.runner>` and
{py:mod}`fsrunner <idaes_fi.structfs.fsrunner>` modules.

### Overview

The core idea of the
{py:class}`FlowsheetRunner <idaes_fi.structfs.fsrunner.FlowsheetRunner>` class is
that flowsheets should follow a standard set of "steps". By standardizing the
naming and ordering of these steps, it becomes easier to build tools that run
and inspect flowsheets. The Python mechanics of this are to put each step in a
function and wrap that function with decorator. The decorator uses a string to
indicate which standard step the function implements.

Once these functions are defined, the API can be used to execute and inspect a
wrapped flowsheet.

The framework can perform arbitrary actions before and after each run,
and before and after a given set of steps. This is implemented with 
the {py:class}`Actions <idaes_fi.structfs.runner.Actions>` class 
and methods `add_action`, `get_action`, and `remove_action` on the base
{py:class}`Runner <idaes_fi.structfs.runner.Runner>` class.
More details are given below in the Actions section.

### Step 1: Define flowsheet

It is assumed here that you have Python code to build, configure, and run an
IDAES flowsheet. You will first arrange this code to follow the standard "steps"
of a flowsheet workflow. 

The set of defined steps is in {py:class}`Steps <idaes_fi.structfs.common.Steps>` class,
as attributes and also in the `index` attribute:

Not all the steps need to be defined: the API will
skip over steps with no definition when executing a range of steps. To make the
code more structured you can also define internal sub-steps.

In [39]:
from idaes_fi.structfs.common import Steps
for i, k in enumerate(Steps.index):  # ordered dict
    print(f"{i +1:2d}) {k}")


 1) build
 2) set_solver
 3) initialize
 4) set_operating_conditions
 5) set_scaling
 6) solve_initial
 7) set_autoscaling
 8) add_costing
 9) initialize_costing
10) setup_optimization
11) solve_optimization


### Example: Flash flowsheet

This is illustrated below with a before/after of an extremely simple flowsheet
with a single Flash unit model.

**Before**

For now, let's assume this flowsheet uses only four of the standard steps:
"build", "set_operating_conditions", "initialize", and "solve_optimization".
Let's also assume you have four functions defined that correspond to these
steps. Below is a sample flowsheet (for a single Flash unit) that we will use as
an example:



In [40]:

from pyomo.environ import ConcreteModel, SolverFactory, SolverStatus
from idaes.core import FlowsheetBlock
from idaes.models.properties.activity_coeff_models.BTX_activity_coeff_VLE import (
    BTXParameterBlock,
)
from idaes.models.unit_models import Flash

def build_model():
    m = ConcreteModel()
    m.fs = FlowsheetBlock(dynamic=False)
    m.fs.properties = BTXParameterBlock(
        valid_phase=("Liq", "Vap"), activity_coeff_model="Ideal", state_vars="FTPz"
    )
    m.fs.flash = Flash(property_package=m.fs.properties)
    return m


def set_operating_conditions(m):
    m.fs.flash.inlet.flow_mol.fix(1)
    m.fs.flash.inlet.temperature.fix(368)
    m.fs.flash.inlet.pressure.fix(101325)
    m.fs.flash.inlet.mole_frac_comp[0, "benzene"].fix(0.5)
    m.fs.flash.inlet.mole_frac_comp[0, "toluene"].fix(0.5)
    m.fs.flash.heat_duty.fix(0)
    m.fs.flash.deltaP.fix(0)


def init_model(m):
    m.fs.flash.initialize()


def solve(m):
    solver = SolverFactory("ipopt")
    return solver.solve(m, tee=True)


#### After

In order to make this into a
{py:class}`FlowsheetRunner <idaes_fi.structfs.fsrunner.FlowsheetRunner>`-wrapped
flowsheet, we need to do make a few changes. The modified file is shown below,
with changed lines highlighted and descriptions below.

In [41]:

from pyomo.environ import ConcreteModel, SolverFactory
from idaes.core import FlowsheetBlock
import idaes.logger as idaeslog
from idaes.models.properties.activity_coeff_models.BTX_activity_coeff_VLE \
import ( BTXParameterBlock, )
from idaes.models.unit_models import Flash

from idaes_fi.structfs.fsrunner import FlowsheetRunner

FS = FlowsheetRunner()

@FS.step("build") 
def build_model(ctx):
    """Build the model."""
    m = ConcreteModel()
    m.fs = FlowsheetBlock(dynamic=False)
    m.fs.properties = BTXParameterBlock(
        valid_phase=("Liq", "Vap"),
        activity_coeff_model="Ideal",
        state_vars="FTPz"
    )
    m.fs.flash = Flash(property_package=m.fs.properties) 
    # assert degrees_of_freedom(m) == 7
    ctx.model = m

@FS.step("set_operating_conditions")
def set_operating_conditions(ctx):
    """Set operating conditions."""
    m = ctx.model
    m.fs.flash.inlet.flow_mol.fix(1)
    m.fs.flash.inlet.temperature.fix(368)
    m.fs.flash.inlet.pressure.fix(101325)
    m.fs.flash.inlet.mole_frac_comp[0, "benzene"].fix(0.5)
    m.fs.flash.inlet.mole_frac_comp[0, "toluene"].fix(0.5)
    m.fs.flash.heat_duty.fix(0)
    m.fs.flash.deltaP.fix(0)

@FS.step("initialize") 
def init_model(ctx): 
    """ "Initialize the model."""
    m = ctx.model
    m.fs.flash.initialize()

@FS.step("set_solver") 
def set_solver(ctx):
    """Set the solver."""
    ctx.solver = SolverFactory("ipopt")

@FS.step("solve_optimization")
def solve_opt(ctx):
    ctx.solve()

### Step 2: Execute and inspect

Once the flowsheet has been 'wrapped' in the flowsheet runner interface, it can
be run and manipulated via the wrapper object. The basic steps to do this are:
import the flowsheet-runner object, build and execute the flowsheet, and inspect
the flowsheet.

For example to run all the steps and get the status of the solve, you
could do this:

In [42]:
FS.run_steps()
assert FS.results.solver.status == "ok"

2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 1 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 2 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 3 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 4 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 5 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_out: Initialization Step 1 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_out: Initialization Step 2 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.prope

----

## Actions 

The Action class implements a simple framework to run arbitrary
functions before and/or after each step and/or run performed
by the `Runner` class.

To create and use your own Action, inherit from this class
and then define one or more of the methods:

* before_step - Called before a given step is executed
* after_step - Called after a given step is executed
* before/after_substep - Called before/after a named
    substep is executed (these can have arbitrary names)
* before_run - Called before the first step is executed
* after_run - Called after the last step is executed

Then add the action to the `Runner` class (e.g., `FlowsheetRunner`)
instance with `add_action()`. Note that you pass the action
*class*, not instance. Additional settings can be passed to
the created action instance with arguments to `add_action`.
Also note that the *name* argument is used to retrieve the
action instance later, as needed.

All actions must also implement the `report()` method,
which returns the results of the action to the caller
as either a Pydantic BaseModel subclass or a Python dict.

### Actions Example

Below is a simple example that prints a message
before/after every step and prints the total number
of steps run at the end of the run.

In [43]:

from idaes_fi.structfs.runner import Action

class HelloGoodbye(Action):
    "Example action, for tutorial purposes."

    def __init__(self, runner, hello="hi", goodbye="bye", **kwargs):
        super().__init__(runner, **kwargs)
        self._hello, self._goodbye = hello, goodbye
        self.step_counter = -1

    def before_run(self):
        self.step_counter = 0

    def before_step(self, name):
        print(f">> {self._hello} from step {name}")

    def before_substep(self, name, subname):
        print(f"  >> {self._hello} from sub-step {subname}")

    def after_step(self, name):
        print(f"<< {self._goodbye} from step {name}")
        self.step_counter += 1

    def after_substep(self, name, subname):
        print(f"  << {self._goodbye} from sub-step {subname}")

    def after_run(self):
        print(f"Ran {self.step_counter} steps")

    def report(self):
        return {"steps": self.step_counter}


You could add the above example to a Runner subclass (`FS`), like this:

In [44]:
_ = FS.add_action(
    "hg",
    HelloGoodbye,
    hello="Greetings and salutations",
    goodbye="Smell you later",
)


    
Then, after running steps, you could print
the value of the `step_counter` attribute:

In [45]:
FS.run_steps()
print(FS.get_action("hg").step_counter)

>> Greetings and salutations from step build
<< Smell you later from step build
<< Smell you later from step set_solver
>> Greetings and salutations from step initialize
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 1 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 2 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 3 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 4 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_in: Initialization Step 5 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idaes.init.fs.flash.control_volume.properties_out: Initialization Step 1 optimal - Optimal Solution Found.
2026-05-02 20:41:31 [INFO] idae


See the pre-defined actions in the
{py:mod}`runner_actions <idaes_fi.structfs.runner_actions>`
module, and their usage in the {py:class}`FlowsheetRunner <idaes_fi.structfs.fsrunner.FlowsheetRunner>` 
class, for more examples.

----

## Annotation

You can also 'annotate' variables for special 
treatment in display, etc. with the
`annotate_var` function in the 
{py:class}`FlowsheetRunner <idaes_fi.structfs.fsrunner.FlowsheetRunner>` class.

### Example

Here is an example of annotating a single Pyomo variable.
You can apply this same technique to any Pyomo, and thus IDAES,
object.


In [46]:
from idaes_fi.structfs.fsrunner import FlowsheetRunner
from pyomo.environ import *

def example(f: FlowsheetRunner):
    v = Var()
    v.construct()
    f.annotate_var(v, key="example", title="Example variable").fix(1)



To retrieve the annotated variables, use the `annotated_vars`
property:

In [47]:
from pprint import pprint

example(fr := FlowsheetRunner())
pprint(fr.annotated_vars)

{'example': {'description': 'ScalarVar',
             'fullname': 'ScalarVar',
             'input_category': 'main',
             'is_input': True,
             'is_output': True,
             'output_category': 'main',
             'rounding': 3,
             'title': 'Example variable',
             'units': 'dimensionless',
             'var': <pyomo.core.base.var.ScalarVar object at 0x79e739a75f60>}}
